# Synthetic query generation

In this notebook, we will generate synthetic pseudo queries based on existing documents.

In [ ]:
import json
import time
import pandas as pd
from datasets import load_dataset
from google import genai
from google.colab import userdata
from tqdm import notebook

## Loading datasets

In [ ]:
# 1. Load the specific dataset. We will use the mteb-nl-news-articles-ret dataset as an example
# The other datasets are called 'clips/mteb-nl-opentender-ret' and 'clips/mteb-nl-vabb-ret' on Huggingface
# Load the three specific components of the dataset
queries_dataset = load_dataset("clips/mteb-nl-news-articles-ret", "queries", split="queries")
corpus_dataset = load_dataset("clips/mteb-nl-news-articles-ret", "corpus", split="corpus")
qrels_dataset = load_dataset("clips/mteb-nl-news-articles-ret", "default", split="test")

In [ ]:
# 2. Convert to Pandas DataFrames for easier data manipulation
df_queries = pd.DataFrame(queries_dataset)  # Columns: _id, text
df_corpus = pd.DataFrame(corpus_dataset)    # Columns: _id,  text
df_qrels = pd.DataFrame(qrels_dataset)      # Columns: query-id, corpus-id, score

# Rename columns for better readability when merging
df_queries = df_queries.rename(columns={'_id': 'query_id', 'text': 'original_query'})
df_corpus = df_corpus.rename(columns={'_id': 'corpus_id', 'text': 'document_text'})
df_qrels = df_qrels.rename(columns={'query-id': 'query_id', 'corpus-id': 'corpus_id'})

In [ ]:
# 3. Merge the dataframes to create a query-document combined dataset
# We first merge the qrels with the original queries
merged_df = pd.merge(df_qrels, df_queries, on='query_id', how='inner')
# Then we merge the merged dataframe with the original corpus
full_dataset_df = pd.merge(merged_df, df_corpus, on='corpus_id', how='inner')

In [ ]:
# # 4. Save the evaluation dataset as a CSV file
# # This contains the paired queries, query-doc relations, and corresponding target texts.
full_dataset_df.to_csv('news_eval.csv', index=False)
# Change CSV name to 'tenders_eval.csv' or 'abstracs_eval.csv'

# # 5. Save the unique document pool as a separate CSV file
# # This represents the total corpus from which the models must retrieve the documents.
df_corpus.to_csv('news_document_pool.csv', index=False)
# Change CSV name to 'tenders_document_pool.csv' or 'abstracts_document_pool.csv'

## Query generation

In [ ]:
# Load the prompt templates and task descriptions
with open("news_prompt.txt", "r", encoding="utf-8") as f:
    prompt_template = f.read()

with open("news_tasks.txt", "r", encoding="utf-8") as f:
    NEWS_TASKS = [task.strip() for task in f.read().split("//")]

# Initialise the Gemini API client
api_key = userdata.get('GEMINI_API_KEY')
client = genai.Client(api_key=api_key)

In [ ]:
def generate_query(document_text, task_description):
    """
    Generates a synthetic query based on a task description and a target document.
    """
    prompt = prompt_template.format(task_description=task_description, text=document_text)

    try:
        response = client.models.generate_content(
            model='gemini-2.5-flash',
            contents=prompt,
            config={
                'response_mime_type': 'application/json',
                'temperature': 0.7
            }
        )

        result = json.loads(response.text)
        return result.get("synthetic_query", "")

    except Exception as e:
        return f"ERROR: {str(e)[:50]}"

In [ ]:
synthetic_queries = []
assigned_tasks = []

for index, row in notebook.tqdm(full_dataset_df.iterrows(), total=len(full_dataset_df)):
    text = row['document_text']

    # Systematically rotate through the available task descriptions
    task_index = index % len(NEWS_TASKS)
    current_task = NEWS_TASKS[task_index]

    #Call API
    query = generate_query(text, current_task)

    synthetic_queries.append(query)
    assigned_tasks.append(f"Task_{task_index + 1}")

    time.sleep(0.05)

# add columns to DataFrame
full_dataset_df['synthetic_query'] = synthetic_queries
full_dataset_df['assigned_task_type'] = assigned_tasks

# Save checkpoint
full_dataset_df.to_csv('news_full_augmented.csv', index=False)
print("✅ Voltooid! Je bestand 'news_full_augmented.csv' staat voor je klaar.")